# 几何探针对比：VGGT 与 3DRAE

**问题：** 同样的探针，读 VGGT 的 token 和读 3DRAE 的 token，哪个能更准地给出物体位置？

**做法：** 探针、数据、真值、损失、训练设置都与 3DRAE 实验相同（见 `README.md` 和 `report_probe.md`），只把被读取的 token 换成 VGGT 的：

- **VGGT token：** 冻结的 VGGT-1B aggregator 最后一轮的 patch token（frame 和 global 两半拼接，2048 维）。VGGT 只看 8 帧图像，不输入相机位姿，以第一帧为参考系，和真值的坐标系一致。
- **相机信息：** 和 DINOv2 对照一样，每个 patch token 加上它的 Plücker 射线（来自真值相机，经可训练线性层）和视图编号嵌入。这样三种 token 拿到的相机信息相同。
- **两种输入分辨率：** 224×308（和 3DRAE 相同，8 帧共 2816 个 token）；378×518（VGGT 训练时的宽度，8 帧共 7992 个 token）。
- **两种查询：** 物体名称提示（类别名的文本嵌入）；点提示（物体中心投影处的 DINOv2 patch 特征加该像素射线，所有 token 来源共用同一种查询）。

**怎么读表：** 物体名称提示统计 EmbodiedScan 标注为可见的物体；点提示统计中心至少落在一帧画面内的物体。验证集 229 个场景，方括号内是按场景 bootstrap 的 95% 置信区间。每组只跑了一个随机种子。

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "outputs" / "geo_probe").is_dir())
RUNS = ROOT / "outputs" / "geo_probe"
pd.set_option("display.width", 200)

CATEGORY = {"3DRAE": "real", "DINOv2 patch": "dino_real", "VGGT 224x308": "vggt_real",
            "VGGT 378x518": "vggt518_real", "shuffled 3DRAE (floor)": "shuffled"}
POINT = {"3DRAE": "point_3drae_real", "DINOv2 patch": "point_dino_real", "VGGT 224x308": "point_vggt_real",
         "VGGT 378x518": "point_vggt518_real", "shuffled 3DRAE (floor)": "point_3drae_shuffled"}
METRICS = [("position_m", "position error (m)", 2), ("distance_m", "pair distance error (m)", 2),
           ("azimuth_deg", "azimuth error (deg)", 1), ("quadrant_acc", "quadrant acc (%)", 1),
           ("lrb_acc", "left/right/back acc (%)", 1)]
COLORS = {"3DRAE": "#2a78d6", "DINOv2 patch": "#8a8f98", "VGGT 224x308": "#eb6834", "VGGT 378x518": "#b23c0c",
          "shuffled 3DRAE (floor)": "#c8ccd2"}


def final(run):
    path = RUNS / run / "final_metrics.json"
    return json.loads(path.read_text()) if path.exists() else None


def table(runs, group):
    rows = {}
    for name, run in runs.items():
        result = final(run)
        if result is None:
            continue
        row = {}
        for key, label, digits in METRICS:
            m = result["eval_real"][f"{group}/{key}"]
            k = 100 if key.endswith("_acc") else 1
            row[label] = f"{m['value'] * k:.{digits}f} [{m['ci95'][0] * k:.{digits}f}, {m['ci95'][1] * k:.{digits}f}]"
        s = result["eval_shuffled"]
        row["tokens swapped at eval: position (m) / quadrant (%)"] = (
            f"{s[f'{group}/position_m']['value']:.2f} / {s[f'{group}/quadrant_acc']['value'] * 100:.1f}")
        row["n objects"] = result["eval_real"][f"{group}/position_m"]["n"]
        rows[name] = row
    return pd.DataFrame(rows).T


def bars(runs, group, title):
    shown = [(n, final(r)) for n, r in runs.items() if final(r)]
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
    for ax, (key, label, _) in zip(axes, [METRICS[0], METRICS[2], METRICS[3]]):
        k = 100 if key.endswith("_acc") else 1
        values = [r["eval_real"][f"{group}/{key}"] for _, r in shown]
        y = np.array([m["value"] * k for m in values])
        err = np.array([[m["value"] - m["ci95"][0], m["ci95"][1] - m["value"]] for m in values]).T * k
        ax.bar(range(len(y)), y, yerr=err, color=[COLORS[n] for n, _ in shown], capsize=3)
        for i, v in enumerate(y):
            ax.text(i, v + err[1][i], f"{v:.2f}" if k == 1 and v < 10 else f"{v:.1f}", ha="center", va="bottom", fontsize=9)
        ax.set_xticks(range(len(y)), [n.replace(" ", "\n", 1) for n, _ in shown], fontsize=8)
        ax.set_title(label, fontsize=10)
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    plt.show()

## 1. 物体名称提示（可见物体）

In [ ]:
category_table = table(CATEGORY, "visible")
category_table

In [ ]:
bars(CATEGORY, "visible", "Category-name query, val visible objects (lower is better for errors)")

## 2. 点提示（中心在画面内的物体）

In [ ]:
point_table = table(POINT, "all")
point_table

In [ ]:
bars(POINT, "all", "Point-prompt query, val objects whose centre is in a view")

## 3. 误差分解

按类别在训练集中的出现次数、物体到第一帧相机的距离分组，看位置误差中位数（米）。只统计中心至少在一帧画面内的物体。如果误差随出现次数明显下降，说明探针受“认出物体”限制；点提示下应当基本持平。

In [ ]:
FREQ = [(0, 0, "0"), (1, 9, "1-9"), (10, 49, "10-49"), (50, 199, "50-199"), (200, 10**9, "200+")]
DIST = [(0, 2, "0-2 m"), (2, 3, "2-3 m"), (3, 4, "3-4 m"), (4, 6, "4-6 m")]


def objects(run):
    path = RUNS / run / "analysis_objects.jsonl"
    if not path.exists():
        return None
    frame = pd.DataFrame([json.loads(line) for line in path.open()])
    return frame[frame.views_seeing_centre > 0]


def breakdown(runs, column, bins, closed_right):
    out = {}
    for name, run in runs.items():
        frame = objects(run)
        if frame is None:
            continue
        row = {}
        for lo, hi, label in bins:
            sel = frame[(frame[column] >= lo) & ((frame[column] <= hi) if closed_right else (frame[column] < hi))]
            row[f"{label} (n={len(sel)})" if name == next(iter(runs)) else label] = sel.error.median()
        out[name] = list(row.values())
        labels = list(row) if name == next(iter(runs)) else labels
    return pd.DataFrame(out, index=labels).T.round(2)


print("position error (m) by category frequency in train")
display(pd.concat({"category query": breakdown(CATEGORY, "train_freq", FREQ, True),
                   "point prompt": breakdown(POINT, "train_freq", FREQ, True)}))
print("position error (m) by object distance from the first camera")
display(pd.concat({"category query": breakdown(CATEGORY, "gt_dist_from_cam0", DIST, False),
                   "point prompt": breakdown(POINT, "gt_dist_from_cam0", DIST, False)}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=False)
for ax, (title, runs) in zip(axes, [("Category-name query", CATEGORY), ("Point prompt", POINT)]):
    for name, run in runs.items():
        frame = objects(run)
        if frame is None:
            continue
        y = [frame[(frame.train_freq >= lo) & (frame.train_freq <= hi)].error.median() for lo, hi, _ in FREQ]
        ax.plot(range(len(FREQ)), y, marker="o", color=COLORS[name], label=name,
                linestyle="--" if "shuffled" in name else "-")
    ax.set_xticks(range(len(FREQ)), [b[2] for b in FREQ])
    ax.set_xlabel("category occurrences in train")
    ax.set_ylabel("median position error (m)")
    ax.set_title(title, fontsize=10)
    ax.set_ylim(bottom=0)
    ax.spines[["top", "right"]].set_visible(False)
axes[1].legend(fontsize=8, frameon=False)
fig.tight_layout()
plt.show()

## 4. 注意力检查（物体名称提示）

DINOv2 和 VGGT 的 token 都对应图像 patch，所以可以看探针第 1 层 cross-attention 落在物体中心投影周围 7×7 个 patch 内的比例，相对均匀分布的倍数。倍数越高，说明探针越能根据名字找到物体。注意 378×518 的 patch 更小，7×7 窗口覆盖的图像范围也更小，和 224×308 的倍数不能直接比。

In [ ]:
rows = {}
for name in ("DINOv2 patch", "VGGT 224x308", "VGGT 378x518"):
    frame = objects(CATEGORY[name])
    if frame is None or "attn_l1_r3" not in frame:
        continue
    frame = frame.dropna(subset=["attn_l1_r3"])
    row = {}
    for lo, hi, label in FREQ:
        sel = frame[(frame.train_freq >= lo) & (frame.train_freq <= hi)]
        row[f"freq {label}"] = sel.attn_l1_r3.mean() / sel.uniform_r3.mean()
    cuts = frame.attn_l1_r3.quantile([1 / 3, 2 / 3]).values
    tertile = np.searchsorted(cuts, frame.attn_l1_r3)
    for k, label in enumerate(("low", "mid", "high")):
        row[f"error (m), {label} attention"] = frame.error[tertile == k].median()
    rows[name] = row
pd.DataFrame(rows).T.round(2)

## 5. 训练曲线

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for ax, (title, runs) in zip(axes, [("Category-name query", CATEGORY), ("Point prompt", POINT)]):
    for name, run in runs.items():
        path = RUNS / run / "metrics.jsonl"
        if not path.exists():
            continue
        log = pd.DataFrame([json.loads(line) for line in path.open()]).drop_duplicates("epoch", keep="last")
        ax.plot(log.epoch, log.loss, color=COLORS[name], label=f"{name} (final {log.loss.iloc[-1]:.3f})",
                linestyle="--" if "shuffled" in name else "-")
    ax.set_yscale("log")
    ax.set_xlabel("epoch")
    ax.set_ylabel("train loss")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

## 6. 结论

（实验完成后填写）